# Day 23 — Prediction API Implementation (manifest-driven)

Your uploaded manifests identify three ML candidate artifacts and three selected deep-learning artifacts. This notebook implements the **ML candidate prediction endpoints first**, because `model_manifest.csv` provides their exact ordered feature lists. It also audits the DL manifest and feature manifests; DL inference is not wired until sequence input shapes and feature JSON contents are confirmed.

## Endpoints
- `POST /api/predict/batting/` — `next_match_runs`
- `POST /api/predict/bowling/` — returns both `next_match_wickets` and `next_match_runs_conceded`
- `POST /api/predict/` — dispatches by `target`

The API accepts the exact features in the manifest and uses the actual saved `.joblib` artifacts. It does not fabricate features from a player name. Existing `views.py` and `urls.py` are preserved; the notebook writes `prediction_views.py` and a URL patch.

**Important:** The manifest paths contain a Windows absolute prefix. This notebook uses the relative suffix under `models/` so the project can run from another machine or AWS deployment. Confirm the files exist in your repository before testing.

In [1]:
from pathlib import Path
import os, sys, subprocess, json, csv, shutil
from datetime import datetime

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    roots = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if roots:
        REPO_ROOT = roots[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
APP_DIR = DJANGO_ROOT / "ipl_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
VIEWS_PATH = APP_DIR / "views.py"
URLS_PATH = APP_DIR / "urls.py"
MODEL_DIR = REPO_ROOT / "models"
REPORT_DIR = REPO_ROOT / "reports" / "day10"
DOCS_DIR = REPO_ROOT / "docs"
assert MANAGE_PY.exists(), f"Run from repository root; missing {MANAGE_PY}"
assert VIEWS_PATH.exists() and URLS_PATH.exists(), "Existing views.py and urls.py must exist."
print("Repository:", REPO_ROOT)
print("Django app:", APP_DIR)


Repository: D:\Git\Personal\ipl-player-performance
Django app: D:\Git\Personal\ipl-player-performance\django_api\ipl_api


In [2]:
# Locate manifests in their original report folders. Do not create a new folder.
import pandas as pd

search_roots = [
    REPO_ROOT / "reports" / "day10",
    REPO_ROOT / "reports" / "day12",
    REPO_ROOT / "reports",
    REPO_ROOT,
]
def find_existing(filename):
    # First check expected canonical location, then search existing project folders.
    preferred = {
        "model_manifest.csv": REPO_ROOT / "reports" / "day10" / filename,
        "model_artifacts_manifest.csv": REPO_ROOT / "reports" / "day12" / filename,
    }.get(filename)
    if preferred and preferred.is_file():
        return preferred
    for root in search_roots:
        if root.exists():
            hits = sorted(root.rglob(filename), key=lambda x: len(x.parts))
            if hits:
                return hits[0]
    return None

ml_manifest_path = find_existing("model_manifest.csv")
dl_manifest_path = find_existing("model_artifacts_manifest.csv")
if ml_manifest_path is None or dl_manifest_path is None:
    raise FileNotFoundError(
        "Manifest files were not found in existing project folders. "
        f"model_manifest.csv={ml_manifest_path}; "
        f"model_artifacts_manifest.csv={dl_manifest_path}. "
        "Do not create a new folder; verify their existing locations."
    )
print("Found ML manifest:", ml_manifest_path)
print("Found DL manifest:", dl_manifest_path)
ml = pd.read_csv(ml_manifest_path)
dl = pd.read_csv(dl_manifest_path)
print("ML manifest:")
display(ml[["dataset", "target", "model_name", "model_path", "feature_columns_json"]])
print("\\nDL manifest:")
display(dl[["dataset", "target", "selected_architecture", "model_path", "imputer_path", "scaler_path", "feature_manifest_path"]])


Found ML manifest: D:\Git\Personal\ipl-player-performance\reports\day10\model_manifest.csv
Found DL manifest: D:\Git\Personal\ipl-player-performance\reports\day12\model_artifacts_manifest.csv
ML manifest:


,dataset,target,model_name,model_path,feature_columns_json
0,batting,next_match_runs,LinearRegression,D:\Git\Personal\ipl-player-performance\models\...,"[""prior_career_batting_runs"", ""prior_last5_mea..."
1,bowling,next_match_wickets,LinearRegression,D:\Git\Personal\ipl-player-performance\models\...,"[""prior_career_bowling_wickets"", ""prior_last5_..."
2,bowling,next_match_runs_conceded,XGBoost,D:\Git\Personal\ipl-player-performance\models\...,"[""prior_career_bowling_wickets"", ""prior_last5_..."


\nDL manifest:


,dataset,target,selected_architecture,model_path,imputer_path,scaler_path,feature_manifest_path
0,batting,next_match_runs,GRU,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...
1,bowling,next_match_wickets,LSTM,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...
2,bowling,next_match_runs_conceded,GRU,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...


In [3]:
# Resolve model paths relative to the repository and verify required files.
def relative_project_path(raw):
    raw = str(raw).strip().replace("\\", "/")
    marker = "models/"
    idx = raw.lower().rfind(marker)
    if idx >= 0:
        return REPO_ROOT / raw[idx:]
    return Path(raw)

checks=[]
for _, row in ml.iterrows():
    p = relative_project_path(row["model_path"])
    features = json.loads(row["feature_columns_json"])
    checks.append({
        "kind":"ML",
        "target":row["target"],
        "model_name":row["model_name"],
        "resolved_path":str(p),
        "exists":p.exists(),
        "feature_count":len(features),
        "features":json.dumps(features),
    })
for _, row in dl.iterrows():
    for field in ["model_path", "imputer_path", "scaler_path", "feature_manifest_path"]:
        p = relative_project_path(row[field])
        checks.append({
            "kind":"DL",
            "target":row["target"],
            "model_name":row["selected_architecture"] + ":" + field,
            "resolved_path":str(p),
            "exists":p.exists(),
            "feature_count":"",
            "features":"",
        })
checks_df = pd.DataFrame(checks)
checks_df.to_csv((REPO_ROOT / "reports" / "day10" / "artifact_path_validation_day23.csv"), index=False)
display(checks_df[["kind","target","model_name","resolved_path","exists","feature_count"]])


,kind,target,model_name,resolved_path,exists,feature_count
0,ML,next_match_runs,LinearRegression,D:\Git\Personal\ipl-player-performance\models\...,True,15
1,ML,next_match_wickets,LinearRegression,D:\Git\Personal\ipl-player-performance\models\...,True,15
2,ML,next_match_runs_conceded,XGBoost,D:\Git\Personal\ipl-player-performance\models\...,True,15
3,DL,next_match_runs,GRU:model_path,D:\Git\Personal\ipl-player-performance\models\...,True,
4,DL,next_match_runs,GRU:imputer_path,D:\Git\Personal\ipl-player-performance\models\...,True,
5,DL,next_match_runs,GRU:scaler_path,D:\Git\Personal\ipl-player-performance\models\...,True,
6,DL,next_match_runs,GRU:feature_manifest_path,D:\Git\Personal\ipl-player-performance\models\...,True,
7,DL,next_match_wickets,LSTM:model_path,D:\Git\Personal\ipl-player-performance\models\...,True,
8,DL,next_match_wickets,LSTM:imputer_path,D:\Git\Personal\ipl-player-performance\models\...,True,
9,DL,next_match_wickets,LSTM:scaler_path,D:\Git\Personal\ipl-player-performance\models\...,True,


In [4]:
# Create an isolated API module using actual model_manifest feature lists.
# Relative paths are stored in the generated config, not machine-specific absolute paths.
model_configs = []
for _, row in ml.iterrows():
    model_configs.append({
        "dataset": str(row["dataset"]),
        "target": str(row["target"]),
        "model_name": str(row["model_name"]),
        "model_relative_path": str(relative_project_path(row["model_path"]).relative_to(REPO_ROOT)).replace("\\", "/"),
        "features": json.loads(row["feature_columns_json"]),
    })

module = r'''import json
from pathlib import Path
import joblib
import numpy as np
from django.conf import settings
from rest_framework import status
from rest_framework.response import Response
from rest_framework.views import APIView

MODEL_CONFIGS = __MODEL_CONFIGS__
_CACHE = {}

def _model_path(config):
    return Path(settings.BASE_DIR).parent / config["model_relative_path"]

def _get_model(config):
    target = config["target"]
    if target not in _CACHE:
        path = _model_path(config)
        if not path.is_file():
            raise FileNotFoundError(f"Model artifact not found for target {target}: {path}")
        _CACHE[target] = joblib.load(path)
    return _CACHE[target]

def _validate_payload(payload, config):
    if not isinstance(payload, dict):
        return None, {"detail": "Request body must be a JSON object."}
    missing = [name for name in config["features"] if name not in payload]
    if missing:
        return None, {"detail": "Missing required model features.", "missing_features": missing}
    bad = []
    values = []
    for name in config["features"]:
        value = payload[name]
        if isinstance(value, bool):
            bad.append(name)
            continue
        try:
            number = float(value)
            if not np.isfinite(number):
                raise ValueError()
            values.append(number)
        except (TypeError, ValueError):
            bad.append(name)
    if bad:
        return None, {"detail": "All model features must be finite numeric values.", "invalid_features": bad}
    return np.asarray([values], dtype=float), None

def _predict(config, payload):
    matrix, error = _validate_payload(payload, config)
    if error:
        return None, error
    model = _get_model(config)
    prediction = model.predict(matrix)
    value = float(np.asarray(prediction).reshape(-1)[0])
    if not np.isfinite(value):
        raise ValueError("Model returned a non-finite prediction.")
    return value, None

class BattingPredictionAPIView(APIView):
    def post(self, request):
        config = next((c for c in MODEL_CONFIGS if c["target"] == "next_match_runs"), None)
        if config is None:
            return Response({"detail": "Batting model is not configured."}, status=status.HTTP_503_SERVICE_UNAVAILABLE)
        try:
            value, error = _predict(config, request.data)
            if error:
                return Response(error, status=status.HTTP_400_BAD_REQUEST)
            return Response({"target": config["target"], "model": config["model_name"], "prediction": value})
        except FileNotFoundError as exc:
            return Response({"detail": str(exc)}, status=status.HTTP_503_SERVICE_UNAVAILABLE)
        except Exception:
            return Response({"detail": "Prediction failed. Check server logs for the underlying error."}, status=status.HTTP_500_INTERNAL_SERVER_ERROR)

class BowlingPredictionAPIView(APIView):
    def post(self, request):
        configs = [c for c in MODEL_CONFIGS if c["target"] in {"next_match_wickets", "next_match_runs_conceded"}]
        if len(configs) != 2:
            return Response({"detail": "Both bowling models must be configured."}, status=status.HTTP_503_SERVICE_UNAVAILABLE)
        results = {}
        try:
            for config in configs:
                value, error = _predict(config, request.data)
                if error:
                    return Response(error, status=status.HTTP_400_BAD_REQUEST)
                results[config["target"]] = {"model": config["model_name"], "prediction": value}
            return Response({"predictions": results})
        except FileNotFoundError as exc:
            return Response({"detail": str(exc)}, status=status.HTTP_503_SERVICE_UNAVAILABLE)
        except Exception:
            return Response({"detail": "Prediction failed. Check server logs for the underlying error."}, status=status.HTTP_500_INTERNAL_SERVER_ERROR)

class UnifiedPredictionAPIView(APIView):
    def post(self, request):
        target = request.data.get("target")
        config = next((c for c in MODEL_CONFIGS if c["target"] == target), None)
        if config is None:
            return Response({"detail": "Unsupported target.", "supported_targets": [c["target"] for c in MODEL_CONFIGS]},
                            status=status.HTTP_400_BAD_REQUEST)
        try:
            value, error = _predict(config, request.data.get("features", request.data))
            if error:
                return Response(error, status=status.HTTP_400_BAD_REQUEST)
            return Response({"target": target, "model": config["model_name"], "prediction": value})
        except FileNotFoundError as exc:
            return Response({"detail": str(exc)}, status=status.HTTP_503_SERVICE_UNAVAILABLE)
        except Exception:
            return Response({"detail": "Prediction failed. Check server logs for the underlying error."}, status=status.HTTP_500_INTERNAL_SERVER_ERROR)
'''
module = module.replace("__MODEL_CONFIGS__", repr(model_configs))
module_path = APP_DIR / "prediction_views.py"
if module_path.exists():
    backup = module_path.with_suffix(".py.day23_backup")
    shutil.copy2(module_path, backup)
    print("Existing prediction_views.py backed up to", backup)
module_path.write_text(module, encoding="utf-8")
compile(module, str(module_path), "exec")
print("Generated:", module_path)
print("ML model configs:", len(model_configs))


Generated: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\prediction_views.py
ML model configs: 3


In [5]:
# Generate URL patch; merge manually into existing ipl_api/urls.py.
patch = '''# Day 23 prediction routes — merge into existing django_api/ipl_api/urls.py
from .prediction_views import (
    BattingPredictionAPIView, BowlingPredictionAPIView, UnifiedPredictionAPIView,
)

# Add to existing urlpatterns:
path("predict/batting/", BattingPredictionAPIView.as_view(), name="predict-batting"),
path("predict/bowling/", BowlingPredictionAPIView.as_view(), name="predict-bowling"),
path("predict/", UnifiedPredictionAPIView.as_view(), name="predict-unified"),
'''
patch_path = APP_DIR / "day23_prediction_urls_patch.txt"
patch_path.write_text(patch, encoding="utf-8")
print("Created URL patch:", patch_path)
print("Do not overwrite existing urls.py. Ensure it already imports path from django.urls.")


Created URL patch: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\day23_prediction_urls_patch.txt
Do not overwrite existing urls.py. Ensure it already imports path from django.urls.


In [6]:
# Run Python syntax check for generated module, Django system check, and Day 21 regression tests.
compile(module_path.read_text(encoding="utf-8"), str(module_path), "exec")
print("prediction_views.py syntax: OK")

def run_cmd(args):
    p = subprocess.run(args, cwd=str(DJANGO_ROOT), text=True, capture_output=True)
    print("$", " ".join(map(str,args)))
    print("Return code:", p.returncode)
    print(p.stdout)
    if p.stderr:
        print(p.stderr)
    return p

django_check = run_cmd([sys.executable, str(MANAGE_PY), "check"])
day21_tests = run_cmd([sys.executable, str(MANAGE_PY), "test", "ipl_api.tests.test_pagination_utils_day21", "--verbosity", "1"])


prediction_views.py syntax: OK
$ c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe D:\Git\Personal\ipl-player-performance\django_api\manage.py check
Return code: 0
System check identified no issues (0 silenced).

$ c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe D:\Git\Personal\ipl-player-performance\django_api\manage.py test ipl_api.tests.test_pagination_utils_day21 --verbosity 1
Return code: 0
Found 6 test(s).
System check identified no issues (0 silenced).

......
----------------------------------------------------------------------
Ran 6 tests in 0.001s

OK



In [7]:
# Write an integration report with actual observed path and check results.
doc = [
    "# Day 23 — Prediction API Implementation",
    "",
    "- Generated: " + datetime.now().isoformat(timespec="seconds"),
    "- ML manifest entries: " + str(len(ml)),
    "- DL manifest entries: " + str(len(dl)),
    "- Django check return code: " + str(django_check.returncode),
    "- Day 21 regression return code: " + str(day21_tests.returncode),
    "",
    "## Model mapping",
    "",
]
for item in model_configs:
    doc.append("- `" + item["target"] + "` → `" + item["model_relative_path"] + "` (" + item["model_name"] + "); features: " + str(len(item["features"])) + ".")
doc += [
    "",
    "## Generated files",
    "",
    "- `django_api/ipl_api/prediction_views.py`",
    "- `django_api/ipl_api/day23_prediction_urls_patch.txt`",
    "- `reports/day10/artifact_path_validation_day23.csv`",
    "",
    "## Limitations",
    "",
    "The generated endpoints use the Day 10 ML candidate artifacts and exact ordered feature lists in model_manifest.csv. They accept numeric model features directly; they do not derive feature values from player_name or database history. Deep-learning inference is not enabled in this module; it requires sequence-window construction matching the Day 11/12 feature manifests.",
    "",
    "The URL patch must be merged manually and the endpoints must be smoke-tested after registration. Do not treat the API as verified until valid and invalid payload tests pass.",
]
(DOCS_DIR / "day23_prediction_api_implementation.md").write_text("\n".join(doc) + "\n", encoding="utf-8")
print("Saved:", DOCS_DIR / "day23_prediction_api_implementation.md")


Saved: D:\Git\Personal\ipl-player-performance\docs\day23_prediction_api_implementation.md


## Final completion checklist

- [ ] All three ML artifacts exist according to `reports/day23/artifact_path_validation.csv`.
- [ ] Inspect the generated `prediction_views.py`.
- [ ] Merge `day23_prediction_urls_patch.txt` into existing `urls.py`.
- [ ] Run `python manage.py check`.
- [ ] Run the Day 21 regression tests.
- [ ] Start the server and test valid feature payloads for all three targets.
- [ ] Test missing features, non-numeric values, unsupported targets, and missing model artifact.
- [ ] Compare API predictions with direct `joblib.load(...).predict(...)` results for identical feature vectors.
- [ ] Implement DL sequence inference separately after verifying feature JSON and sequence shapes.

**This is a concrete ML prediction API integration; Day 23 is not complete until URL registration and runtime smoke tests pass.**